# AI Governance Risk-Screening Agent

Most teams plug AI into a workflow without a structured check on whether it's actually a risky use case. This notebook is a small two-agent pipeline that does a first-pass risk screen: one agent assesses a use case across five risk dimensions, and a second agent reviews that assessment critically before producing the final memo.

The five dimensions are grounded in the NIST AI Risk Management Framework.

**Setup before running:** get an API key from console.anthropic.com, then in the left sidebar of this notebook click the key icon, add a new secret named `ANTHROPIC_API_KEY`, paste your key as the value, and toggle notebook access on.

In [1]:
!pip install anthropic -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 932.0/932.0 kB 12.9 MB/s eta 0:00:00


## The pipeline

Two functions: `assess_risk` scores the use case across five dimensions, and `critique_and_finalize` reviews that assessment for anything missed or overconfident, then writes the final memo.

In [2]:
from google.colab import userdata
import os
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

import anthropic
client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY automatically

RUBRIC = """
Assess the AI use case below across these 5 dimensions. For each, give a risk
level (Low/Medium/High) and a 1-2 sentence rationale.

1. Bias & fairness — could this discriminate across protected groups?
2. Privacy & data governance — does it use sensitive personal data, and how is it handled?
3. Transparency & explainability — can a decision be explained to the person it affects?
4. Human oversight & accountability — does a human actually review outputs before they take effect?
5. Regulatory exposure — does this touch a regulated domain (employment, healthcare, finance, housing)?
"""

def assess_risk(use_case):
    response = client.messages.create(
        model="claude-sonnet-4-6", max_tokens=1000,
        messages=[{"role": "user", "content": f"{RUBRIC}\n\nUse case:\n{use_case}"}]
    )
    return response.content[0].text

def critique_and_finalize(use_case, assessment):
    prompt = f"""You are reviewing this risk assessment for accuracy. Be skeptical.

Use case: {use_case}
Assessment to review: {assessment}

First, briefly note anything missed or overconfident. Then write a final
one-page memo: overall risk rating, per-dimension findings, 2-3 prioritized
mitigations, and a closing line noting this is a first-pass screening, not a
compliance or legal determination."""
    response = client.messages.create(
        model="claude-sonnet-4-6", max_tokens=1500,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text

## Test case 1: the obvious one — but obvious doesn't mean simple

AI resume screening is the textbook high-risk use case. Running it first establishes the ceiling: what does an all-red scorecard actually look like, and why?

In [3]:
use_case = "We want an LLM to automatically screen job applications and rank candidates before a recruiter sees them."

assessment = assess_risk(use_case)
print(assessment)

# AI Risk Assessment: Automated Job Application Screening & Ranking

---

## 1. Bias & Fairness
**Risk Level: 🔴 HIGH**

LLMs trained on historical data can encode and amplify societal biases related to gender, race, age, or socioeconomic background — for example, favoring certain name patterns, educational institutions, or writing styles. Automated ranking before any human review means discriminatory patterns can systematically exclude protected groups at scale before anyone notices.

---

## 2. Privacy & Data Governance
**Risk Level: 🔴 HIGH**

Job applications contain highly sensitive personal data including contact details, employment history, education, and potentially protected characteristics. Feeding this into an LLM raises serious questions about data retention, third-party model provider data handling, consent, and compliance with GDPR or state-level privacy laws.

---

## 3. Transparency & Explainability
**Risk Level: 🔴 HIGH**

LLM ranking decisions are largely a black box — i

In [4]:
final_memo = critique_and_finalize(use_case, assessment)
print(final_memo)

## Initial Critique

**Missed or underweighted issues:**

- **Job description quality risk** is absent. If the input prompt encoding the job criteria is poorly designed, the LLM will optimize for the wrong attributes regardless of bias controls. Garbage-in amplified at scale.
- **Gaming and adversarial inputs** aren't mentioned. Candidates who learn to write for AI screeners gain an advantage unrelated to actual job fit, which is both a fairness and quality-of-hire problem.
- **Model hallucination and factual errors** are unaddressed. An LLM may misread, misparse, or confabulate details from a resume (e.g., misattributing dates, inferring qualifications not stated), affecting rankings in ways that are hard to detect.
- **Scope of "eliminated before human review"** could be more precise. The assessment conflates ranking with hard-cutoff exclusion. These carry different risk profiles — ranking with full human review is materially lower risk than a hard discard threshold.

**Overconfidenc

## Test case 2: the control case

An internal coding assistant feels harmless. The critic agent disagrees — not dramatically, but precisely. This is what a genuinely low-medium risk profile looks like and why the distinctions matter.

In [5]:
use_case_2 = "An internal coding assistant that suggests code completions to engineers as they write code."

assessment_2 = assess_risk(use_case_2)
print(assessment_2)

# AI Risk Assessment: Internal Coding Assistant for Engineers

---

## 1. Bias & Fairness
**Risk Level: Low**
The tool suggests code completions to engineers, a function that does not make decisions about people across protected characteristics. Any bias in suggestions (e.g., favoring certain coding patterns or libraries) has minimal discriminatory impact on individuals.

---

## 2. Privacy & Data Governance
**Risk Level: Medium**
If the assistant ingests internal codebases, it may inadvertently process proprietary logic, credentials, API keys, or PII embedded in code. Governance controls are needed to ensure sensitive data is not transmitted to third-party model providers or retained in training pipelines.

---

## 3. Transparency & Explainability
**Risk Level: Low**
Engineers are technical users who can read, evaluate, and understand suggested code before accepting it. No opaque decision is imposed on an affected party — the suggestion is visible and interpretable by design.

---

##

In [6]:
final_memo_2 = critique_and_finalize(use_case_2, assessment_2)
print(final_memo_2)

## Initial Skeptical Review: What's Missed or Overconfident

**Bias & Fairness — possibly too dismissive.** The tool shapes engineering habits at scale. Systematic bias toward certain languages, frameworks, or coding patterns could entrench technical debt, disadvantage engineers unfamiliar with the favored patterns, or subtly de-skill teams over time. "No decisions about people" is too narrow a frame.

**Human Oversight — overconfident.** The "engineers review everything" argument assumes careful, skeptical review every time. In practice, autocomplete suggestions are accepted quickly and habitually. Velocity pressure makes rubber-stamping likely. The PR/code review layer is real but also often cursory, and reviewers may not flag AI-suggested code differently than human-written code. The assessment treats a theoretical safeguard as a reliable one.

**Regulatory Exposure — the "downstream concern" dismissal is too casual.** If engineers are writing code *for* a regulated system (healthca

## Test case 3: life-critical AI in a high-volume environment

An ER triage assistant that recommends patient priority levels before a nurse reviews them — and where oversight erodes precisely when it matters most.

In [9]:
# Test case 3: a medium-high risk use case in clinical AI
# An ER triage assistant that reads patient-reported symptoms and vitals,
# then recommends a priority level before a nurse reviews the patient.

use_case_3 = """A hospital emergency department deploys an LLM-based triage assistant.
When a patient checks in, they enter their symptoms, pain level (1-10), and basic vitals
(heart rate, blood pressure) into a tablet at reception. The LLM reads this input and
assigns a priority level — Immediate, Urgent, or Non-Urgent — which is displayed on the
triage nurse's dashboard. The nurse sees this recommendation before conducting their own
assessment, but during high-volume periods the recommendation is often accepted without
independent re-evaluation."""

assessment_3 = assess_risk(use_case_3)
print(assessment_3)

# AI Triage Assistant – Risk Assessment

---

## 1. Bias & Fairness
**Risk Level: HIGH**

LLMs trained on historical medical data can encode systemic disparities — for example, well-documented undertriage of pain in Black patients or women — and self-reported pain scores are themselves subject to cultural communication differences. A miscalibrated priority label accepted without re-evaluation could systematically delay care for already-disadvantaged groups.

---

## 2. Privacy & Data Governance
**Risk Level: HIGH**

The system collects real-time health data (symptoms, vitals, pain levels) entered directly by patients, constituting protected health information (PHI) under HIPAA. Key concerns include how this data is transmitted to and processed by the LLM, whether it is retained or used for model training, and whether appropriate data processing agreements are in place — especially if a third-party LLM provider is involved.

---

## 3. Transparency & Explainability
**Risk Level: HIGH**


In [10]:
final_memo_3 = critique_and_finalize(use_case_3, assessment_3)
print(final_memo_3)

## Initial Critique

**Missed or underweighted issues:**

- **Adversarial/data quality risk** is absent entirely. Patients can misreport symptoms deliberately (drug-seeking, malingering) or accidentally (health literacy, language barriers, pain at time of entry). The LLM has no way to validate self-reported inputs, and a triage system built on unverified self-report is more fragile than the assessment acknowledges.
- **System failure and availability risk** is not addressed. What happens when the tablet or LLM is unavailable? If staff have habituated to relying on the recommendation, the fallback protocol matters enormously.
- **Anchoring bias in the nurse** deserves more explicit treatment. Seeing a recommendation *before* independent assessment is not merely a workflow gap — it is a documented cognitive hazard (anchoring/automation bias) that degrades the quality of the nominally independent review even when the nurse *does* re-evaluate.
- **LLM-specific failure modes** unique to thi

## Test case 4: automation bias hiding in plain sight

A support ticket router where productivity incentives make human review illusory — agents are technically in the loop, but the system is functionally auto-deciding.

In [11]:
# Test case 4: a medium-risk use case in customer operations
# An AI tool that summarizes support tickets and suggests a response category
# before the agent reads the ticket.

use_case_4 = """A SaaS company deploys an LLM to process incoming customer support tickets.
When a ticket arrives, the LLM reads it and outputs a one-paragraph summary and a suggested
response category: Refund, Escalate, or Resolve. The support agent sees the summary and
category recommendation before reading the original ticket. Agents are expected to review
the full ticket before acting, but productivity metrics reward speed, and agents are
evaluated on tickets closed per hour."""

assessment_4 = assess_risk(use_case_4)
print(assessment_4)

# AI Risk Assessment: LLM-Powered Customer Support Triage

---

## 1. Bias & Fairness
**Risk Level: Medium**

The LLM may systematically miscategorize tickets written in non-standard English, by non-native speakers, or using culturally specific phrasing, potentially routing those customers to less favorable outcomes (e.g., "Resolve" instead of "Escalate"). Because the agent sees the AI summary *before* the original ticket, anchor bias may cause agents to unconsciously favor the AI's framing, amplifying any underlying model bias at scale.

---

## 2. Privacy & Data Governance
**Risk Level: Medium**

Customer support tickets routinely contain sensitive personal information — account details, payment information, health or personal circumstances offered as context. The key risks are whether this data is being sent to a third-party LLM API, how long it is retained, and whether it is used for model training. These questions require clear data processing agreements and should be audited.

--

In [12]:
final_memo_4 = critique_and_finalize(use_case_4, assessment_4)
print(final_memo_4)

## Initial Skeptical Review: Gaps and Overconfidence

**Missed or underweighted issues:**

- **Summary hallucination risk is absent.** The assessment never addresses the possibility that the LLM's one-paragraph summary may contain fabrications or material omissions — not just bias, but factually wrong content. An agent anchoring on a hallucinated summary is a distinct and serious failure mode from bias.
- **Feedback loop / model drift is not mentioned.** If agent actions are logged and fed back into model evaluation or fine-tuning, systematically rubber-stamped decisions corrupt the training signal over time.
- **The "Resolve" category deserves more scrutiny.** "Resolve" likely means closing the ticket without escalation or refund. The assessment treats the three categories roughly equally, but wrongful "Resolve" routing is the highest-consequence error and deserves its own treatment.
- **Vendor / supply chain risk is underexplored.** The privacy section mentions third-party APIs but d

##Try your own use case

Describe any AI deployment scenario below and the pipeline will assess it across all five risk dimensions and produce a final risk memo.

Tips for a useful output:
- Be specific about who uses the system and how
- Mention whether a human reviews outputs before they take effect
- Include any productivity pressures or incentive structures that affect how carefully people review AI recommendations

In [ ]:
my_use_case = """Describe your AI use case here. Be specific —
who uses it, what decisions it influences, and whether a human
reviews outputs before they take effect."""

my_assessment = assess_risk(my_use_case)
my_final_memo = critique_and_finalize(my_use_case, my_assessment)
print(my_final_memo)